# Minilab EduBI — Eksplorasi Data

Notebook ini digunakan untuk eksplorasi interaktif data dari DuckDB.

**Prasyarat**: Sudah jalankan ETL dan dbt terlebih dahulu.

```bash
docker compose run --rm app
docker compose run --rm dbt
```

In [ ]:
import duckdb
import pandas as pd

# Buka koneksi ke DuckDB (read-only)
con = duckdb.connect('../data/warehouse/lab_bi.duckdb', read_only=True)
print('Terhubung ke DuckDB.')

In [ ]:
# Lihat semua tabel yang tersedia
con.execute("SHOW ALL TABLES").fetchdf()

## Bronze Layer

In [ ]:
# Preview bronze.sales
con.execute("SELECT * FROM bronze.sales LIMIT 5").fetchdf()

In [ ]:
# Jumlah baris per tabel bronze
for tbl in ['sales', 'customers', 'reviews', 'targets']:
    n = con.execute(f"SELECT COUNT(*) FROM bronze.{tbl}").fetchone()[0]
    print(f"bronze.{tbl}: {n} baris")

## Silver Layer

In [ ]:
# Preview silver.silver_sales
con.execute("SELECT * FROM silver.silver_sales LIMIT 5").fetchdf()

In [ ]:
# Distribusi revenue_category
con.execute("""
    SELECT revenue_category, COUNT(*) AS jumlah
    FROM silver.silver_sales
    GROUP BY revenue_category
    ORDER BY jumlah DESC
""").fetchdf()

In [ ]:
# Distribusi sentimen ulasan
con.execute("""
    SELECT sentiment, COUNT(*) AS jumlah
    FROM silver.silver_reviews
    GROUP BY sentiment
    ORDER BY jumlah DESC
""").fetchdf()

## Gold Layer

In [ ]:
# KPI per cabang
con.execute("""
    SELECT
        branch,
        total_orders,
        total_revenue,
        avg_order_value,
        revenue_achievement_pct,
        avg_rating
    FROM gold.gold_branch_kpi
    ORDER BY total_revenue DESC
""").fetchdf()

In [ ]:
# Tren revenue per bulan
con.execute("""
    SELECT
        order_year,
        order_month,
        SUM(total_revenue) AS monthly_revenue,
        SUM(total_orders)  AS monthly_orders
    FROM gold.gold_sales_daily
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""").fetchdf()

In [ ]:
# Ringkasan ulasan per cabang
con.execute("""
    SELECT branch, avg_rating, total_reviews, pct_positif
    FROM gold.gold_review_summary
    ORDER BY avg_rating DESC
""").fetchdf()

In [ ]:
con.close()
print('Koneksi ditutup.')